# Experimentos con una ResNet simple en CIFAR-10

Este notebook contiene algunos experimentos y análisis con la implementación propia de una **ResNet pequeña** definida en `src/resnet_keras.py`.

## Contenido

1. Descripción breve de la arquitectura.
2. Carga del modelo y de los pesos entrenados.
3. Evaluación en el conjunto de test.
4. Visualización de métricas y curvas de entrenamiento.
5. Inspección de predicciones y análisis de errores.

> Nota: este notebook está pensado para mostrar tu entendimiento del modelo y los resultados, **no es el notebook original del curso de Andrew Ng**.

In [4]:
import tensorflow as tf
from pathlib import Path
import sys
import os

from matplotlib import pyplot as plt

sys.path.append(os.path.abspath("../src"))

from resnet_keras import build_resnet50

DATA_DIR = Path("../data")
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Cargar pesos de archivo

In [ ]:
# Cargar modelo y pesos entrenados (si existen)

# Nota: la arquitectura exacta (tamaño de entrada, nº de clases) se
# recupera del archivo .h5, así que este modelo inicial es solo de referencia.
# CIFAR-10 se reescala a 64x64 para usar la arquitectura exacta del notebook original.
model = build_resnet50(input_shape=(64, 64, 3), classes=10)

# Buscamos el archivo de pesos tanto en esta carpeta como en la raíz del proyecto
local_weights = Path("small_resnet_keras.h5")
root_weights = Path("../small_resnet_keras.h5")

if local_weights.exists():
    weights_path = local_weights
elif root_weights.exists():
    weights_path = root_weights
else:
    weights_path = None

if weights_path is not None:
    model = tf.keras.models.load_model(weights_path)
    print(f"Modelo cargado desde {weights_path}")
else:
    print("No se encontraron pesos entrenados. Ejecuta una de las celdas de entrenamiento de abajo para crear el archivo de pesos.")

Modelo cargado desde small_resnet_keras.h5


## Entrenar con dataset de Keras CIFAR-10

In [ ]:
from train_keras import train_with_cifar
history = train_with_cifar(epochs=2)

from tensorflow.keras.models import load_model
model = load_model(weights_path)
print("Modelo entrenado y cargado desde:", weights_path)

## Evaluación rápida en test

En esta sección puedes:

- Calcular la `accuracy` en el conjunto de test.
- Visualizar algunas imágenes y sus predicciones.
- Comentar los casos donde la red falla.

La idea es **explicar con tus palabras** qué está haciendo bien/mal la ResNet, y cómo se relaciona eso con la profundidad del modelo y las skip connections.

In [ ]:
# Evaluar el modelo en el conjunto de test de CIFAR-10

# Cargamos CIFAR-10 desde Keras
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalizamos y reescalamos a 64x64 (igual que en el entrenamiento)
x_test = x_test.astype("float32") / 255.0
x_test = tf.image.resize(x_test, (64, 64))

# One-hot encoding de las etiquetas
num_classes = 10
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Evaluación
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

Test loss: 1.3994
Test accuracy: 0.6111
